# NeTheR on AWA2: Animals problem 1

This notebook runs NeTheR on the first saved Animals problem, `2_2_1_0_expert.json`. It loads the prepared **Animals with Attributes 2 (AWA2)** subset, visualizes the data and theories, revises the expert theory, and evaluates the result.

Original dataset: [Animals with Attributes 2](https://cvml.ista.ac.at/AwA2/). The original release contains 37,322 images in 50 classes with 85 class-level attributes. The prepared data in this repository uses up to 100 images per class, resized to 64 x 64 RGB. Refer to the source page for the dataset citation and per-image licenses.

The saved problem contains 3,999 training, 500 validation, and 501 test examples. This run uses the requested settings: `edit_distance=5`, `k=3`, `freeze=True`, `symb_allowed=True`, and at most 75 epochs per neural training run. Because `k=3` trains three calibration networks before search, a CUDA GPU is strongly recommended.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import f1_score, precision_score, recall_score

from nether.Data.dataloaders import animal_image, load_small_animal_data
from nether.Problems.problem_loader import load_problem
from nether.src.Main import approximate_revision
from nether.src.propositional import load_theory
from nether.src.train_model import validate_model

np.random.seed(10)
torch.manual_seed(10)
DEVICE = [0] if torch.cuda.is_available() else None
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'PyTorch {torch.__version__} | compute device: {device_name}')

## Configuration

The paper's main evaluation used `k=3`, froze previously learned neural concepts, and used an 80%/10%/10% data split. Early stopping stops training after 5 validation epochs without improvement.

In [ ]:
PROBLEM_FILE = '2_2_1_0_expert.json'
EDIT_DISTANCE = 5
K = 3
FREEZE = True
SYMB_ALLOWED = True
NEURAL_ALLOWED = True
EPOCHS = 50
MODE = 'animal_conv'

pd.Series({
    'problem': PROBLEM_FILE,
    'edit_distance': EDIT_DISTANCE,
    'k': K,
    'freeze': FREEZE,
    'symb_allowed': SYMB_ALLOWED,
    'neural_allowed': NEURAL_ALLOWED,
    'epochs': EPOCHS,
    'device': device_name,
}, name='value').to_frame()

## Load and inspect the AWA2 data

Each row contains 85 symbolic animal attributes followed by a flattened 64 x 64 RGB image. Labels for this binary task are generated by the saved ground-truth theory rather than by the animal class itself.

In [ ]:
full_data, categorical_columns, _, animal_names = load_small_animal_data(
    return_animal_names=True
)
problem = load_problem('animals', PROBLEM_FILE)
training_data = full_data.loc[problem['train_indices']].copy()
validation_data = full_data.loc[problem['validation_indices']].copy()
test_data = full_data.loc[problem['test_indices']].copy()
training_labels = problem['train_labels']
validation_labels = problem['validation_labels']
test_labels = problem['test_labels']

split_summary = pd.DataFrame({
    'examples': [len(training_data), len(validation_data), len(test_data)],
    'positive': [sum(training_labels), sum(validation_labels), sum(test_labels)],
}, index=['train', 'validation', 'test'])
split_summary['positive_fraction'] = (
    split_summary['positive'] / split_summary['examples']
).round(3)
display(split_summary)
print(f'Feature matrix: {full_data.shape[0]:,} rows x {full_data.shape[1]:,} columns')

## Expert and ground-truth theories

The ground-truth theory is $M^*$, used to generate the binary labels. The expert theory $M'$ is imperfect because the `newworld` literal was removed. NeTheR receives only $M'$ and the labeled training/validation data; $M^*$ is displayed for evaluation and interpretation only.

In [ ]:
ground_truth_theory = load_theory(problem['original_query'])
expert_theory = load_theory(problem['expert_query'])

def theory_expression(node):
    if node.is_var():
        return node.condition.replace('==', '=')
    if node.is_neural_predicate():
        return node.name()
    operator = ' AND ' if node.is_and() else ' OR '
    return '(' + operator.join(theory_expression(child) for child in node.children) + ')'

display(Markdown('### Ground truth, $M^*$'))
print(theory_expression(ground_truth_theory))
print('\nCircuit representation:\n' + ground_truth_theory.show())

display(Markdown("### Expert theory, $M'$"))
print(theory_expression(expert_theory))
print('\nCircuit representation:\n' + expert_theory.show())
print('Removed ground-truth attributes:', problem['dropped_columns'])

In [ ]:
train_label = dict(zip(problem['train_indices'], training_labels))
positive_examples = [i for i in problem['train_indices'] if train_label[i] == 1][:3]
negative_examples = [i for i in problem['train_indices'] if train_label[i] == 0][:3]
example_indices = positive_examples + negative_examples

preview = full_data.loc[example_indices, ['oldworld', 'newworld', 'strainteeth', 'furry']].copy()
preview.insert(0, 'animal', animal_names.loc[example_indices].to_numpy())
preview['target'] = [train_label[i] for i in example_indices]
display(preview)

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, index in zip(axes.flat, example_indices):
    ax.imshow(animal_image(full_data, index))
    ax.set_title(f'{animal_names.loc[index]} | target={train_label[index]}')
    ax.axis('off')
fig.suptitle('Three positive and three negative training examples')
fig.tight_layout()

In [ ]:
def symbolic_predictions(theory, data):
    covered = set(theory.get_covering_examples(data, device=DEVICE))
    return [int(index in covered) for index in data.index]

expert_validation_predictions = symbolic_predictions(expert_theory, validation_data)
expert_test_predictions = symbolic_predictions(expert_theory, test_data)

baseline = pd.DataFrame({
    'validation': [f1_score(validation_labels, expert_validation_predictions)],
    'test': [f1_score(test_labels, expert_test_predictions)],
}, index=['expert F1'])
baseline.round(3)

## Run NeTheR

This is the expensive cell. On its first neural insertion, TorchVision may download the pretrained ResNet-101 weights. NeTheR first trains up to `k=3` candidate neural concepts to calibrate its Sharpe-ratio estimate and then performs up to five accepted revisions.

In [ ]:
def condition_column(condition):
    for operator in ('<=', '>=', '==', '<', '>'):
        if operator in condition:
            return condition.split(operator, 1)[0].strip()
    raise ValueError(f'Unsupported condition: {condition}')

allowed_symbolic_columns = sorted({
    condition_column(variable.condition)
    for variable in expert_theory.variables()
})
print('Symbolic columns available to NeTheR:', allowed_symbolic_columns)

# Reload a clean copy because theory objects cache coverage information.
expert_for_revision = load_theory(problem['expert_query'])
started = time.perf_counter()
best_result, timing, operations_path = approximate_revision(
    data=(training_data, validation_data),
    theory=expert_for_revision,
    labels=(training_labels, validation_labels),
    cat_cols=allowed_symbolic_columns,
    edit_distance=EDIT_DISTANCE,
    k=K,
    mode=MODE,
    freeze=FREEZE,
    epochs=EPOCHS,
    device=DEVICE,
    symb_allowed=SYMB_ALLOWED,
    neural_allowed=NEURAL_ALLOWED,
    test_data=test_data,
    test_labels=test_labels,
)
elapsed_seconds = time.perf_counter() - started
print(f'NeTheR completed in {elapsed_seconds / 60:.1f} minutes')

## Evaluate the revised theory

In [ ]:
validation_f1, revised_model, revised_theory = best_result
if revised_model is None:
    revised_test_predictions = symbolic_predictions(revised_theory, test_data)
else:
    revised_test_predictions = validate_model(
        revised_model, test_data, device=DEVICE
    )

metrics = pd.DataFrame({
    'expert theory': [
        f1_score(test_labels, expert_test_predictions),
        precision_score(test_labels, expert_test_predictions, zero_division=0),
        recall_score(test_labels, expert_test_predictions, zero_division=0),
    ],
    'revised theory': [
        f1_score(test_labels, revised_test_predictions),
        precision_score(test_labels, revised_test_predictions, zero_division=0),
        recall_score(test_labels, revised_test_predictions, zero_division=0),
    ],
}, index=['F1', 'precision', 'recall'])
display(metrics.round(3))

display(Markdown('### Revised theory'))
print(theory_expression(revised_theory))
print('\nCircuit representation:\n' + revised_theory.show())

In [ ]:
path_columns = ['operation', 'test_F1', 'validation_F1', 'alpha']
display(pd.DataFrame(operations_path, columns=path_columns).round(3))

timing_summary = {name: sum(values) for name, values in timing.items()}
timing_summary['total_wall_clock'] = elapsed_seconds
pd.Series(timing_summary, name='seconds').round(1).to_frame()